In [1]:
# Instalando o que não vem por padrão no Colab
!pip install openmeteo-requests requests-cache retry-requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.5/772.5 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.3 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


In [2]:
import requests
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta
import time

# ⚠️ Coloque sua chave da NASA aqui
NASA_API_KEY = "68067347ab0ec0f021417cc98bbb1acf"

# Região do Cerrado brasileiro (bounding box)
# lat_min, lat_max, lon_min, lon_max
CERRADO = {
    "lat_min": -20.0,
    "lat_max": -10.0,
    "lon_min": -52.0,
    "lon_max": -44.0
}

print("✅ Configurações carregadas!")

✅ Configurações carregadas!


In [3]:
def buscar_focos_nasa(ano, mes):
    """
    Busca focos de calor no Cerrado para um mês específico.
    Faz requisições de 5 em 5 dias (limite da API).
    """
    import calendar
    from io import StringIO

    area = f"{CERRADO['lon_min']},{CERRADO['lat_min']},{CERRADO['lon_max']},{CERRADO['lat_max']}"
    dias_no_mes = calendar.monthrange(ano, mes)[1]

    dfs_mes = []

    # Percorre o mês de 5 em 5 dias
    dia = 1
    while dia <= dias_no_mes:
        data_str = f"{ano}-{mes:02d}-{dia:02d}"

        url = (
            f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
            f"{NASA_API_KEY}/MODIS_SP/{area}/5/{data_str}"
        )

        try:
            response = requests.get(url, timeout=60)

            if response.status_code == 200:
                df = pd.read_csv(StringIO(response.text))
                if len(df) > 0:
                    dfs_mes.append(df)

        except Exception as e:
            print(f"  ❌ Erro em {data_str}: {e}")

        dia += 5
        time.sleep(0.5)  # pausa entre requisições

    if dfs_mes:
        df_resultado = pd.concat(dfs_mes, ignore_index=True)
        df_resultado['ano_mes'] = f"{ano}-{mes:02d}"
        print(f"  ✅ {ano}-{mes:02d}: {len(df_resultado)} focos encontrados")
        return df_resultado
    else:
        print(f"  ⚠️ {ano}-{mes:02d}: sem focos detectados")
        return pd.DataFrame()

print("✅ Função atualizada!")

✅ Função atualizada!


In [4]:
print("🛰️ Iniciando coleta de dados da NASA FIRMS...\n")

todos_focos = []

for ano in range(2020, 2025):  # 2020, 2021, 2022, 2023, 2024
    for mes in range(1, 13):   # janeiro a dezembro
        df_mes = buscar_focos_nasa(ano, mes)
        if not df_mes.empty:
            todos_focos.append(df_mes)
        time.sleep(1)  # pausa para não sobrecarregar a API

# Junta tudo num único DataFrame
df_focos = pd.concat(todos_focos, ignore_index=True)

print(f"\n📊 Total de registros coletados: {len(df_focos)}")
print(df_focos.head())

🛰️ Iniciando coleta de dados da NASA FIRMS...

  ✅ 2020-01: 622 focos encontrados
  ✅ 2020-02: 531 focos encontrados
  ✅ 2020-03: 950 focos encontrados
  ✅ 2020-04: 907 focos encontrados
  ✅ 2020-05: 1663 focos encontrados
  ✅ 2020-06: 2673 focos encontrados
  ✅ 2020-07: 4357 focos encontrados
  ✅ 2020-08: 5889 focos encontrados
  ✅ 2020-09: 15317 focos encontrados
  ✅ 2020-10: 14254 focos encontrados
  ✅ 2020-11: 2126 focos encontrados
  ✅ 2020-12: 1042 focos encontrados
  ✅ 2021-01: 815 focos encontrados
  ✅ 2021-02: 297 focos encontrados
  ✅ 2021-03: 811 focos encontrados
  ✅ 2021-04: 661 focos encontrados
  ✅ 2021-05: 2794 focos encontrados
  ❌ Erro em 2021-06-01: Invalid input ConnectionInputs.SEND_HEADERS in state ConnectionState.CLOSED
  ✅ 2021-06: 2965 focos encontrados
  ✅ 2021-07: 6844 focos encontrados
  ✅ 2021-08: 13699 focos encontrados
  ✅ 2021-09: 24258 focos encontrados
  ✅ 2021-10: 7281 focos encontrados
  ✅ 2021-11: 553 focos encontrados
  ✅ 2021-12: 297 focos encontr

In [5]:
import requests

area = f"{CERRADO['lon_min']},{CERRADO['lat_min']},{CERRADO['lon_max']},{CERRADO['lat_max']}"

url = (
    f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{NASA_API_KEY}/MODIS_NRT/{area}/1/2023-08-01"
)

print("URL gerada:")
print(url)
print()

response = requests.get(url, timeout=30)

print(f"Status HTTP: {response.status_code}")
print()
print("Resposta da NASA:")
print(response.text[:500])

URL gerada:
https://firms.modaps.eosdis.nasa.gov/api/area/csv/68067347ab0ec0f021417cc98bbb1acf/MODIS_NRT/-52.0,-20.0,-44.0,-10.0/1/2023-08-01

Status HTTP: 200

Resposta da NASA:
latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight


In [6]:
df_focos.to_csv("focos_cerrado_2020_2024.csv", index=False)
print("💾 Arquivo salvo: focos_cerrado_2020_2024.csv")
print(f"Colunas disponíveis: {list(df_focos.columns)}")

💾 Arquivo salvo: focos_cerrado_2020_2024.csv
Colunas disponíveis: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'ano_mes']


In [7]:
def buscar_clima_openmeteo(latitude, longitude, data_inicio, data_fim):
    """
    Busca dados climáticos históricos para uma coordenada específica.
    Open-Meteo é gratuito e não precisa de chave de API.
    """
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": data_inicio,  # formato: "2020-01-01"
        "end_date": data_fim,       # formato: "2024-12-31"
        "daily": [
            "temperature_2m_max",       # temperatura máxima
            "precipitation_sum",         # chuva acumulada
            "windspeed_10m_max",        # velocidade do vento
            "relative_humidity_2m_max"  # umidade máxima
        ]
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]

    # Extrai os dados diários
    daily = response.Daily()
    df_clima = pd.DataFrame({
        "data": pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s"),
            end=pd.to_datetime(daily.TimeEnd(), unit="s"),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        ),
        "latitude": latitude,
        "longitude": longitude,
        "temp_max": daily.Variables(0).ValuesAsNumpy(),
        "precipitacao": daily.Variables(1).ValuesAsNumpy(),
        "vento_max": daily.Variables(2).ValuesAsNumpy(),
        "umidade_max": daily.Variables(3).ValuesAsNumpy(),
    })

    return df_clima

# Testando com o centro do Cerrado (Brasília como referência)
print("🌤️ Buscando dados climáticos históricos...")
df_clima = buscar_clima_openmeteo(
    latitude=-15.78,
    longitude=-47.93,
    data_inicio="2020-01-01",
    data_fim="2024-12-31"
)

print(f"✅ {len(df_clima)} dias de dados climáticos coletados")
print(df_clima.head())

🌤️ Buscando dados climáticos históricos...
✅ 1827 dias de dados climáticos coletados
        data  latitude  longitude   temp_max  precipitacao  vento_max  \
0 2020-01-01    -15.78     -47.93  25.950001     20.800001   9.504272   
1 2020-01-02    -15.78     -47.93  26.000000     20.100000  15.716793   
2 2020-01-03    -15.78     -47.93  25.350000     16.100002  17.786331   
3 2020-01-04    -15.78     -47.93  24.049999      8.599999  22.183128   
4 2020-01-05    -15.78     -47.93  23.600000     21.500002  21.129883   

   umidade_max  
0    95.409309  
1    95.434929  
2    96.920860  
3    95.133575  
4    95.725517  


In [8]:
df_clima.to_csv("clima_cerrado_2020_2024.csv", index=False)
print("💾 Arquivo salvo: clima_cerrado_2020_2024.csv")

💾 Arquivo salvo: clima_cerrado_2020_2024.csv


In [9]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

# Carrega os arquivos locais
df_focos = pd.read_csv("focos_cerrado_2020_2024.csv")
df_clima = pd.read_csv("clima_cerrado_2020_2024.csv")

print(f"✅ Focos carregados: {len(df_focos)} registros")
print(f"✅ Clima carregado: {len(df_clima)} registros")

# Cria a pasta no Drive se não existir
os.makedirs("/content/drive/MyDrive/agroguard", exist_ok=True)

# Salva no Drive
df_focos.to_csv("/content/drive/MyDrive/agroguard/focos_cerrado_2020_2024.csv", index=False)
df_clima.to_csv("/content/drive/MyDrive/agroguard/clima_cerrado_2020_2024.csv", index=False)

print("💾 Arquivos salvos no Google Drive!")

ValueError: mount failed